In [ ]:
import os
from pathlib import Path

import torch
from hydra import compose, initialize
from hydra.utils import instantiate
from lightning import LightningDataModule
from omegaconf import OmegaConf

from nicheflow.tasks import FlowMatching
from nicheflow.utils import print_config

OmegaConf.register_new_resolver("add", lambda x, y: x + y)

# We need this because the checkpoints of the classifiers are currently
# using relative paths rather than absolute paths => We need to be
# in the root nicheflow directory.
os.chdir("../")
print("Current working directory:", os.getcwd())

In [ ]:
initialize(config_path="../configs", version_base=None)

In [ ]:
experiment_template = "{model}/{variant}/{dataset}"

In [ ]:
ot_lambda_override = 0.1
experiment = experiment_template.format(model="nicheflow", variant="glvfm", dataset="med")


# For OT lambda = 0.1
ckpt_root_path = Path("ckpts")
ckpt_path = ckpt_root_path.joinpath("NicheFlow_GLVFM_MED.ckpt")

# For OT lambda = [0.25, 0.5, 0.75]
# ckpt_path = ckpt_root_path.joinpath(
#     "ot_ablations", f"NicheFlow_GLVFM_MED_OTLambda={ot_lambda_override}.ckpt"
# )

config = compose(
    config_name="train",
    overrides=[
        f"experiment={experiment}",
        # For OT lambda = [0.25, 0.5, 0.75]
        # f"data.datamodule.ot_lambda={ot_lambda_override}"
    ],
)

# Overwrite these paths to deal with the hydra instance issue.
config.paths.output_dir = Path("outputs")
config.paths.work_dir = config.paths.root_dir

OmegaConf.resolve(config)
print_config(config)

In [ ]:
datamodule: LightningDataModule = instantiate(config.data.datamodule)

# This will also load the classifier checkpoint
model: FlowMatching = instantiate(config.model)

# Prep the data
datamodule.prepare_data()
datamodule.setup("test")

# Load the state dictionary checkpoint
ckpt = torch.load(
    ckpt_path,
    weights_only=False,
    map_location="cpu",
)
model.flow.backbone.load_state_dict(ckpt)

In [ ]:
test_dl = datamodule.test_dataloader()

In [ ]:
from tqdm import tqdm

model = model.to(torch.device("cuda"))


def batch_to_cuda(batch: dict[str, torch.Tensor]) -> dict[str, torch.Tensor]:
    return {key: value.to(torch.device("cuda")) for key, value in batch.items()}


model.eval()
with torch.no_grad():
    for batch in tqdm(test_dl, "Iterating"):
        X_traj, pos_traj = model.flow.sample(batch_to_cuda(batch))

In [ ]:
X_traj[-1].shape, pos_traj[-1].shape